### Project Solution: Goals 1 and 2

In this notebook we will build the solution step by step.

The project asks us to refactor the previous `Polygon` and `Polygons` classes.

We have two main goals:

* make the computed `Polygon` properties lazy
* make `Polygons` a lazy iterable, using a separate iterator class

We will use plain `assert` statements for our tests, just like we would do while learning or prototyping.


### What lazy means here

A lazy property is not computed when the object is created.

Instead, it is computed the first time we ask for it.

After that, the computed value is stored in a private variable and reused.

For example:

```python
p = Polygon(4, 1)
```

At this point, the area should not be computed yet.

But after this:

```python
p.area
```

the area should be computed once and stored.


### Goal 1

We need to refactor the `Polygon` class.

Each polygon is a regular convex polygon centered at `(0, 0)`.

The initializer needs:

* `n`: the number of vertices, minimum `3`
* `R`: the circumradius, meaning the distance from the center to every vertex

All the other values can be computed from `n` and `R`.


For a regular polygon with `n` vertices and circumradius `R`, we will use these formulas:

* interior angle: `(n - 2) * 180 / n`
* side length: `2 * R * sin(pi / n)`
* apothem: `R * cos(pi / n)`
* perimeter: `n * side_length`
* area: `1 / 2 * perimeter * apothem`

We will also add a useful ratio:

```python
area / perimeter
```

This is the efficiency value we will use later for the `Polygons` iterable.


Let's start by importing what we need.

We need `math` for the polygon formulas.

We also need `islice` later to show that iteration is lazy.


In [1]:
import math
from itertools import islice


Before writing the class, let's create two small validation helpers.

This keeps the validation logic simple and avoids repeating the same checks in both `Polygon` and `Polygons`.


In [2]:
def validate_vertex_count(value, name='n'):
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError(f'{name} must be an integer')

    if value < 3:
        raise ValueError(f'{name} must be at least 3')


def validate_radius(value, name='R'):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f'{name} must be a real number')

    if not math.isfinite(value) or value <= 0:
        raise ValueError(f'{name} must be a positive finite number')


Now let's create our `Polygon` class.

The important part for Goal 1 is this pattern:

```python
self._area = None
```

Then inside the property:

```python
if self._area is None:
    self._area = ...
return self._area
```

Since we also want the object to behave as immutable, we will use `object.__setattr__` inside the lazy properties.

That lets the class update its own private cache, while normal external attribute changes are still blocked.


In [3]:
class Polygon:
    __slots__ = (
        '_n',
        '_R',
        '_interior_angle',
        '_side_length',
        '_apothem',
        '_perimeter',
        '_area',
        '_area_to_perimeter_ratio',
        '_vertex_coordinates',
        '_locked'
    )

    def __init__(self, n, R):
        validate_vertex_count(n, 'n')
        validate_radius(R, 'R')

        object.__setattr__(self, '_n', n)
        object.__setattr__(self, '_R', R)

        # Lazy backing variables.
        object.__setattr__(self, '_interior_angle', None)
        object.__setattr__(self, '_side_length', None)
        object.__setattr__(self, '_apothem', None)
        object.__setattr__(self, '_perimeter', None)
        object.__setattr__(self, '_area', None)
        object.__setattr__(self, '_area_to_perimeter_ratio', None)
        object.__setattr__(self, '_vertex_coordinates', None)

        # Lock the instance after initialization.
        object.__setattr__(self, '_locked', True)

    def __setattr__(self, name, value):
        if getattr(self, '_locked', False):
            raise AttributeError('Polygon objects are immutable.')
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError('Polygon objects are immutable.')

    def __repr__(self):
        return f'Polygon(n={self._n}, R={self._R})'

    @property
    def count_vertices(self):
        return self._n

    @property
    def num_vertices(self):
        return self._n

    @property
    def count_edges(self):
        return self._n

    @property
    def num_edges(self):
        return self._n

    @property
    def count_sides(self):
        return self._n

    @property
    def circumradius(self):
        return self._R

    @property
    def interior_angle(self):
        if self._interior_angle is None:
            value = (self._n - 2) * 180 / self._n
            object.__setattr__(self, '_interior_angle', value)
        return self._interior_angle

    @property
    def side_length(self):
        if self._side_length is None:
            value = 2 * self._R * math.sin(math.pi / self._n)
            object.__setattr__(self, '_side_length', value)
        return self._side_length

    @property
    def edge_length(self):
        return self.side_length

    @property
    def apothem(self):
        if self._apothem is None:
            value = self._R * math.cos(math.pi / self._n)
            object.__setattr__(self, '_apothem', value)
        return self._apothem

    @property
    def perimeter(self):
        if self._perimeter is None:
            value = self._n * self.side_length
            object.__setattr__(self, '_perimeter', value)
        return self._perimeter

    @property
    def area(self):
        if self._area is None:
            value = self.perimeter * self.apothem / 2
            object.__setattr__(self, '_area', value)
        return self._area

    @property
    def area_to_perimeter_ratio(self):
        if self._area_to_perimeter_ratio is None:
            value = self.area / self.perimeter
            object.__setattr__(self, '_area_to_perimeter_ratio', value)
        return self._area_to_perimeter_ratio

    @property
    def vertex_coordinates(self):
        if self._vertex_coordinates is None:
            value = tuple(
                (
                    self._R * math.cos(2 * math.pi * vertex / self._n),
                    self._R * math.sin(2 * math.pi * vertex / self._n)
                )
                for vertex in range(self._n)
            )
            object.__setattr__(self, '_vertex_coordinates', value)
        return self._vertex_coordinates

    def as_dict(self):
        return {
            'n': self._n,
            'R': self._R,
            'interior_angle': self.interior_angle,
            'side_length': self.side_length,
            'apothem': self.apothem,
            'area': self.area,
            'perimeter': self.perimeter,
            'area_to_perimeter_ratio': self.area_to_perimeter_ratio
        }

    def __eq__(self, other):
        if isinstance(other, Polygon):
            return (
                self.count_vertices == other.count_vertices
                and self.circumradius == other.circumradius
            )
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices < other.count_vertices
        return NotImplemented

    def __le__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices <= other.count_vertices
        return NotImplemented

    def __gt__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices > other.count_vertices
        return NotImplemented

    def __ge__(self, other):
        if isinstance(other, Polygon):
            return self.count_vertices >= other.count_vertices
        return NotImplemented


Let's create a small helper for testing exceptions.

This keeps our tests readable.


In [4]:
def assert_raises(exception_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except exception_type:
        return
    except Exception as ex:
        assert False, f'expected {exception_type.__name__}, but got {type(ex).__name__}'
    else:
        assert False, f'expected {exception_type.__name__}, but no exception was raised'


Let's test the basic properties first.

Notice that if an assertion is true, nothing is displayed.

If an assertion is false, Python raises an `AssertionError`.


In [5]:
def test_polygon_basic_properties():
    n = 3
    R = 1
    p = Polygon(n, R)

    assert str(p) == 'Polygon(n=3, R=1)', f'actual: {str(p)}'
    assert p.count_vertices == n, f'actual: {p.count_vertices}, expected: {n}'
    assert p.num_vertices == n, f'actual: {p.num_vertices}, expected: {n}'
    assert p.count_edges == n, f'actual: {p.count_edges}, expected: {n}'
    assert p.num_edges == n, f'actual: {p.num_edges}, expected: {n}'
    assert p.count_sides == n, f'actual: {p.count_sides}, expected: {n}'
    assert p.circumradius == R, f'actual: {p.circumradius}, expected: {R}'
    assert p.interior_angle == 60, f'actual: {p.interior_angle}, expected: 60'


In [6]:
test_polygon_basic_properties()


Next we will test some computed values.

For a square with circumradius `1`:

* side length should be `sqrt(2)`
* perimeter should be `4 * sqrt(2)`
* apothem should be `sqrt(2) / 2`
* area should be `2`

When testing floats, we should not use exact equality.

Instead, we will use `math.isclose`.


In [7]:
def test_polygon_square_values():
    abs_tol = 0.001
    rel_tol = 0.001

    p = Polygon(4, 1)

    assert p.interior_angle == 90, f'actual: {p.interior_angle}, expected: 90'

    assert math.isclose(
        p.side_length, math.sqrt(2),
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.side_length}, expected: {math.sqrt(2)}'

    assert math.isclose(
        p.edge_length, math.sqrt(2),
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.edge_length}, expected: {math.sqrt(2)}'

    assert math.isclose(
        p.perimeter, 4 * math.sqrt(2),
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.perimeter}, expected: {4 * math.sqrt(2)}'

    assert math.isclose(
        p.apothem, math.sqrt(2) / 2,
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.apothem}, expected: {math.sqrt(2) / 2}'

    assert math.isclose(
        p.area, 2,
        rel_tol=rel_tol, abs_tol=abs_tol
    ), f'actual: {p.area}, expected: 2'


In [8]:
test_polygon_square_values()


Let's test a few more known values.

For `n = 6`, `R = 2`:

* side length is approximately `2`
* apothem is approximately `1.73205`
* area is approximately `10.3923`
* perimeter is approximately `12`
* interior angle is `120`

For `n = 12`, `R = 3`:

* side length is approximately `1.55291`
* apothem is approximately `2.89778`
* area is approximately `27`
* perimeter is approximately `18.635`
* interior angle is `150`


In [9]:
def test_polygon_more_values():
    abs_tol = 0.001
    rel_tol = 0.001

    p = Polygon(6, 2)

    assert math.isclose(p.side_length, 2, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 1.73205, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 10.3923, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 12, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 120, rel_tol=rel_tol, abs_tol=abs_tol)

    p = Polygon(12, 3)

    assert math.isclose(p.side_length, 1.55291, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 2.89778, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 27, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 18.635, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 150, rel_tol=rel_tol, abs_tol=abs_tol)


In [10]:
test_polygon_more_values()


Now let's test validation.

We want to reject invalid vertex counts and invalid radii.


In [11]:
def test_polygon_validation():
    assert_raises(ValueError, Polygon, 2, 1)
    assert_raises(TypeError, Polygon, 3.5, 1)
    assert_raises(TypeError, Polygon, True, 1)

    assert_raises(ValueError, Polygon, 3, 0)
    assert_raises(ValueError, Polygon, 3, -1)
    assert_raises(ValueError, Polygon, 3, float('inf'))
    assert_raises(ValueError, Polygon, 3, float('nan'))
    assert_raises(TypeError, Polygon, 3, '10')


In [12]:
test_polygon_validation()


Now let's test equality and ordering.

The project requires:

* equality based on number of vertices and circumradius
* ordering based on number of vertices only


In [13]:
def test_polygon_equality_and_ordering():
    p1 = Polygon(3, 10)
    p2 = Polygon(10, 10)
    p3 = Polygon(15, 10)
    p4 = Polygon(15, 100)
    p5 = Polygon(15, 100)

    assert p2 > p1
    assert p2 < p3

    assert p3 != p4
    assert p1 != p4
    assert p4 == p5

    # Ordering uses only the number of vertices.
    assert p3 >= p4
    assert p3 <= p4


In [14]:
test_polygon_equality_and_ordering()


Now let's test immutability.

We should not be able to change `_n`, change `_R`, add a new attribute, or delete an attribute after the object has been created.


In [15]:
def test_polygon_immutability():
    p = Polygon(4, 1)

    assert_raises(AttributeError, setattr, p, '_n', 10)
    assert_raises(AttributeError, setattr, p, '_R', 100)
    assert_raises(AttributeError, setattr, p, 'new_attribute', 'not allowed')
    assert_raises(AttributeError, delattr, p, '_n')


In [16]:
test_polygon_immutability()


Now let's test that the computed properties are actually lazy.

Before accessing the properties, their backing variables should be `None`.

After accessing a property, the matching backing variable should have a value.


In [17]:
def test_polygon_lazy_properties():
    p = Polygon(6, 2)

    assert p._interior_angle is None
    assert p._side_length is None
    assert p._apothem is None
    assert p._perimeter is None
    assert p._area is None
    assert p._area_to_perimeter_ratio is None
    assert p._vertex_coordinates is None

    side_length = p.side_length

    assert p._side_length is not None
    assert p.side_length == side_length

    # Accessing area will also need perimeter, side length, and apothem.
    area = p.area

    assert p._area is not None
    assert p._perimeter is not None
    assert p._side_length is not None
    assert p._apothem is not None
    assert p.area == area

    ratio = p.area_to_perimeter_ratio

    assert p._area_to_perimeter_ratio is not None
    assert p.area_to_perimeter_ratio == ratio

    points = p.vertex_coordinates

    assert p._vertex_coordinates is not None
    assert p.vertex_coordinates is points


In [18]:
test_polygon_lazy_properties()


Let's also check our small useful extra: vertex coordinates.

For a square with circumradius `1`, the first point starts at angle `0`, so it should be `(1, 0)`.


In [19]:
def test_polygon_vertex_coordinates():
    p = Polygon(4, 1)
    points = p.vertex_coordinates

    assert len(points) == 4

    x, y = points[0]
    assert math.isclose(x, 1, abs_tol=0.001)
    assert math.isclose(y, 0, abs_tol=0.001)

    for x, y in points:
        distance_from_center = math.sqrt(x ** 2 + y ** 2)
        assert math.isclose(distance_from_center, 1, abs_tol=0.001)


In [20]:
test_polygon_vertex_coordinates()


At this point, Goal 1 is complete.

The `Polygon` class has lazy computed properties.

Now we can move on to Goal 2.


### Goal 2

We need to refactor `Polygons`.

Previously, it stored a list like this:

```python
self._polygons = [Polygon(i, R) for i in range(3, m + 1)]
```

But the project now asks us to avoid that.

The new version should be lazy:

* it should not create all polygons during initialization
* it should create each polygon only when that polygon is requested during iteration
* it should use a separate iterator class


The iterable object and iterator object are different things.

The iterable is the object we loop over:

```python
polygons = Polygons(6, 10)
```

The iterator is the object that keeps track of where we are in the loop:

```python
iterator = iter(polygons)
next(iterator)
```

Every call to `iter(polygons)` should return a new independent iterator.


Let's start with the iterator.

The iterator only needs three pieces of state:

* the current number of vertices
* the maximum number of vertices
* the shared circumradius

Each call to `next()` will create one `Polygon`.


In [21]:
class PolygonsIterator:
    __slots__ = ('_current_n', '_max_n', '_R')

    def __init__(self, max_n, R):
        self._current_n = 3
        self._max_n = max_n
        self._R = R

    def __iter__(self):
        return self

    def __next__(self):
        if self._current_n > self._max_n:
            raise StopIteration

        polygon = Polygon(self._current_n, self._R)
        self._current_n += 1
        return polygon


Now let's create the `Polygons` iterable.

This class will not store a list of polygons.

It only stores:

* `m`, the maximum number of vertices
* `R`, the circumradius
* a lazy backing variable for the most efficient polygon

We will also keep a few useful sequence-like features such as indexing and slicing.

Those features are still lazy because we calculate polygon objects from a `range`, not from a stored list.


In [22]:
class Polygons:
    __slots__ = ('_m', '_R', '_max_efficiency_polygon', '_locked')

    def __init__(self, m, R):
        validate_vertex_count(m, 'm')
        validate_radius(R, 'R')

        object.__setattr__(self, '_m', m)
        object.__setattr__(self, '_R', R)

        # Lazy backing variable.
        object.__setattr__(self, '_max_efficiency_polygon', None)

        object.__setattr__(self, '_locked', True)

    def __setattr__(self, name, value):
        if getattr(self, '_locked', False):
            raise AttributeError('Polygons objects are immutable.')
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError('Polygons objects are immutable.')

    def __repr__(self):
        return f'Polygons(m={self._m}, R={self._R})'

    @property
    def max_vertices(self):
        return self._m

    @property
    def circumradius(self):
        return self._R

    @property
    def length(self):
        return len(self)

    def __len__(self):
        return self._m - 2

    def __iter__(self):
        return PolygonsIterator(self._m, self._R)

    def __getitem__(self, index):
        vertex_numbers = range(3, self._m + 1)

        if isinstance(index, slice):
            return tuple(Polygon(n, self._R) for n in vertex_numbers[index])

        if isinstance(index, bool) or not isinstance(index, int):
            raise TypeError('indices must be integers or slices')

        try:
            n = vertex_numbers[index]
        except IndexError:
            raise IndexError('polygon index out of range')

        return Polygon(n, self._R)

    def __contains__(self, polygon):
        return (
            isinstance(polygon, Polygon)
            and polygon.circumradius == self._R
            and 3 <= polygon.count_vertices <= self._m
        )

    @property
    def max_efficiency_polygon(self):
        if self._max_efficiency_polygon is None:
            value = max(self, key=lambda polygon: polygon.area_to_perimeter_ratio)
            object.__setattr__(self, '_max_efficiency_polygon', value)
        return self._max_efficiency_polygon

    @property
    def max_area_perimeter_ratio_polygon(self):
        return self.max_efficiency_polygon

    @property
    def most_efficient_polygon(self):
        return self.max_efficiency_polygon

    def ratios(self):
        return tuple(p.area_to_perimeter_ratio for p in self)

    def summary(self):
        return tuple(p.as_dict() for p in self)


Let's test the basic iterable behavior first.


In [23]:
def test_polygons_basic_iterable():
    polygons = Polygons(6, 10)

    assert str(polygons) == 'Polygons(m=6, R=10)', f'actual: {str(polygons)}'
    assert len(polygons) == 4
    assert polygons.length == 4
    assert polygons.max_vertices == 6
    assert polygons.circumradius == 10

    assert tuple(polygons) == (
        Polygon(3, 10),
        Polygon(4, 10),
        Polygon(5, 10),
        Polygon(6, 10)
    )


In [24]:
test_polygons_basic_iterable()


Now let's test the iterator protocol.

The iterator should be its own iterator.

That means:

```python
iter(iterator) is iterator
```

But the iterable should return a new iterator each time.

That means:

```python
iter(polygons) is not polygons
```


In [25]:
def test_polygons_iterator_protocol():
    polygons = Polygons(6, 10)

    iterator = iter(polygons)

    assert isinstance(iterator, PolygonsIterator)
    assert iterator is not polygons
    assert iter(iterator) is iterator

    assert next(iterator) == Polygon(3, 10)
    assert next(iterator) == Polygon(4, 10)

    assert list(iterator) == [
        Polygon(5, 10),
        Polygon(6, 10)
    ]

    assert_raises(StopIteration, next, iterator)


In [26]:
test_polygons_iterator_protocol()


Now let's test that separate iterators are independent.

Advancing one iterator should not advance another iterator.


In [27]:
def test_polygons_independent_iterators():
    polygons = Polygons(6, 10)

    iterator_1 = iter(polygons)
    iterator_2 = iter(polygons)

    assert next(iterator_1) == Polygon(3, 10)
    assert next(iterator_1) == Polygon(4, 10)

    assert next(iterator_2) == Polygon(3, 10)
    assert next(iterator_2) == Polygon(4, 10)
    assert next(iterator_2) == Polygon(5, 10)

    assert next(iterator_1) == Polygon(5, 10)


In [28]:
test_polygons_independent_iterators()


Now let's test that `Polygons` does not store a list of polygons.

We also use a very large upper bound and only take the first few polygons.

If the class tried to build all polygons first, this would be wasteful.


In [29]:
def test_polygons_lazy_iteration():
    polygons = Polygons(1_000_000, 1)

    assert not hasattr(polygons, '_polygons')

    first_three = tuple(islice(polygons, 3))

    assert first_three == (
        Polygon(3, 1),
        Polygon(4, 1),
        Polygon(5, 1)
    )


In [30]:
test_polygons_lazy_iteration()


Indexing is not required for a basic iterable, but it is a useful extra.

Since we can calculate the needed number of vertices directly, indexing still does not require a stored list.


In [31]:
def test_polygons_indexing():
    polygons = Polygons(6, 10)

    assert polygons[0] == Polygon(3, 10)
    assert polygons[1] == Polygon(4, 10)
    assert polygons[2] == Polygon(5, 10)
    assert polygons[3] == Polygon(6, 10)

    assert polygons[-1] == Polygon(6, 10)
    assert polygons[-2] == Polygon(5, 10)
    assert polygons[-3] == Polygon(4, 10)
    assert polygons[-4] == Polygon(3, 10)


In [32]:
test_polygons_indexing()


Now let's test invalid indexing.

Out-of-range indices should raise `IndexError`.

Non-integer indices should raise `TypeError`.


In [33]:
def test_polygons_invalid_indexing():
    polygons = Polygons(6, 10)

    assert_raises(IndexError, polygons.__getitem__, 4)
    assert_raises(IndexError, polygons.__getitem__, -5)
    assert_raises(TypeError, polygons.__getitem__, 1.5)
    assert_raises(TypeError, polygons.__getitem__, True)


In [34]:
test_polygons_invalid_indexing()


Now let's test slicing.

Slicing returns a tuple of polygons.

This still does not use an internal list; the polygons are created from the relevant part of a `range`.


In [35]:
def test_polygons_slicing():
    polygons = Polygons(8, 1)

    assert polygons[0:3] == (
        Polygon(3, 1),
        Polygon(4, 1),
        Polygon(5, 1)
    )

    assert polygons[2:] == (
        Polygon(5, 1),
        Polygon(6, 1),
        Polygon(7, 1),
        Polygon(8, 1)
    )

    assert polygons[:2] == (
        Polygon(3, 1),
        Polygon(4, 1)
    )


In [36]:
test_polygons_slicing()


Extended slicing should also work.

That includes things like:

```python
polygons[::2]
polygons[::-1]
```


In [37]:
def test_polygons_extended_slicing():
    polygons = Polygons(10, 1)

    assert polygons[::2] == (
        Polygon(3, 1),
        Polygon(5, 1),
        Polygon(7, 1),
        Polygon(9, 1)
    )

    assert polygons[1:7:2] == (
        Polygon(4, 1),
        Polygon(6, 1),
        Polygon(8, 1)
    )

    assert polygons[::-1] == (
        Polygon(10, 1),
        Polygon(9, 1),
        Polygon(8, 1),
        Polygon(7, 1),
        Polygon(6, 1),
        Polygon(5, 1),
        Polygon(4, 1),
        Polygon(3, 1)
    )


In [38]:
test_polygons_extended_slicing()


Let's also test membership.

A polygon belongs to the iterable if:

* it is a `Polygon`
* it has the same circumradius
* its number of vertices is between `3` and `m`


In [39]:
def test_polygons_membership():
    polygons = Polygons(6, 3)

    assert Polygon(3, 3) in polygons
    assert Polygon(6, 3) in polygons
    assert Polygon(7, 3) not in polygons
    assert Polygon(6, 4) not in polygons
    assert 'not a polygon' not in polygons


In [40]:
test_polygons_membership()


Now let's test the highest `area / perimeter` ratio.

The property itself is lazy too.

Before we access it, `_max_efficiency_polygon` should be `None`.

After we access it, it should contain the best polygon.


In [41]:
def test_polygons_best_ratio():
    polygons = Polygons(12, 5)

    assert polygons._max_efficiency_polygon is None

    expected = max(polygons, key=lambda p: p.area_to_perimeter_ratio)
    actual = polygons.max_efficiency_polygon

    assert actual == expected
    assert actual == Polygon(12, 5)
    assert polygons.max_area_perimeter_ratio_polygon == Polygon(12, 5)
    assert polygons.most_efficient_polygon == Polygon(12, 5)

    # The property is cached.
    assert polygons.max_efficiency_polygon is actual


In [42]:
test_polygons_best_ratio()


Let's test `Polygons` validation and immutability too.


In [43]:
def test_polygons_validation_and_immutability():
    assert_raises(ValueError, Polygons, 2, 1)
    assert_raises(TypeError, Polygons, 3.5, 1)
    assert_raises(TypeError, Polygons, True, 1)

    assert_raises(ValueError, Polygons, 3, 0)
    assert_raises(ValueError, Polygons, 3, -1)
    assert_raises(ValueError, Polygons, 3, float('inf'))
    assert_raises(ValueError, Polygons, 3, float('nan'))
    assert_raises(TypeError, Polygons, 3, '10')

    polygons = Polygons(8, 2)

    assert_raises(AttributeError, setattr, polygons, '_m', 100)
    assert_raises(AttributeError, setattr, polygons, '_R', 100)
    assert_raises(AttributeError, setattr, polygons, 'new_attribute', 'not allowed')
    assert_raises(AttributeError, delattr, polygons, '_m')


In [44]:
test_polygons_validation_and_immutability()


Finally, let's test the small extra utility methods.

The `summary()` method returns a tuple of dictionaries, one dictionary per polygon.

The `ratios()` method returns all area-to-perimeter ratios.


In [45]:
def test_polygons_extra_utilities():
    polygons = Polygons(5, 1)

    ratios = polygons.ratios()
    summary = polygons.summary()

    assert len(ratios) == len(polygons)
    assert len(summary) == len(polygons)

    assert isinstance(summary[0], dict)
    assert summary[0]['n'] == 3
    assert summary[-1]['n'] == 5

    assert ratios == tuple(p.area_to_perimeter_ratio for p in polygons)


In [46]:
test_polygons_extra_utilities()


Let's run all the tests together.

This gives us one clean cell that can be re-run whenever we make changes.


In [47]:
def run_all_tests():
    test_polygon_basic_properties()
    test_polygon_square_values()
    test_polygon_more_values()
    test_polygon_validation()
    test_polygon_equality_and_ordering()
    test_polygon_immutability()
    test_polygon_lazy_properties()
    test_polygon_vertex_coordinates()

    test_polygons_basic_iterable()
    test_polygons_iterator_protocol()
    test_polygons_independent_iterators()
    test_polygons_lazy_iteration()
    test_polygons_indexing()
    test_polygons_invalid_indexing()
    test_polygons_slicing()
    test_polygons_extended_slicing()
    test_polygons_membership()
    test_polygons_best_ratio()
    test_polygons_validation_and_immutability()
    test_polygons_extra_utilities()

    print('All tests passed!')


In [48]:
run_all_tests()


All tests passed!


### Example usage

Let's create a lazy iterable of polygons from triangles to octagons with circumradius `3`.


In [49]:
polygons = Polygons(8, 3)

print(polygons)
print('length:', len(polygons))
print('first polygon:', polygons[0])
print('last polygon:', polygons[-1])
print('every other polygon:', polygons[::2])
print('best ratio polygon:', polygons.max_efficiency_polygon)


Polygons(m=8, R=3)
length: 6
first polygon: Polygon(n=3, R=3)
last polygon: Polygon(n=8, R=3)
every other polygon: (Polygon(n=3, R=3), Polygon(n=5, R=3), Polygon(n=7, R=3))
best ratio polygon: Polygon(n=8, R=3)


And let's print a small table manually using formatted strings.


In [50]:
print(f'{"n":>3} {"side":>10} {"apothem":>10} {"area":>10} {"perimeter":>12} {"ratio":>10}')
print('-' * 61)

for p in polygons:
    print(
        f'{p.count_vertices:>3} '
        f'{p.side_length:>10.4f} '
        f'{p.apothem:>10.4f} '
        f'{p.area:>10.4f} '
        f'{p.perimeter:>12.4f} '
        f'{p.area_to_perimeter_ratio:>10.4f}'
    )


  n       side    apothem       area    perimeter      ratio
-------------------------------------------------------------
  3     5.1962     1.5000    11.6913      15.5885     0.7500
  4     4.2426     2.1213    18.0000      16.9706     1.0607
  5     3.5267     2.4271    21.3988      17.6336     1.2135
  6     3.0000     2.5981    23.3827      18.0000     1.2990
  7     2.6033     2.7029    24.6277      18.2231     1.3515
  8     2.2961     2.7716    25.4558      18.3688     1.3858


### Final notes

The solution now satisfies the project requirements:

* `Polygon` computed properties are lazy
* `Polygon` objects behave as immutable objects
* `Polygons` does not store a pre-built list of polygons
* `Polygons` is an iterable
* `PolygonsIterator` is a separate iterator class
* each polygon is created only when requested during iteration
* every call to `iter(polygons)` creates a fresh independent iterator
* the most efficient polygon is also cached lazily
* tests cover the important behavior and edge cases

We also added a few useful extras:

* aliases such as `count_sides`, `num_vertices`, and `edge_length`
* `vertex_coordinates`
* indexing and slicing without internal list storage
* membership checks
* summary and ratio helpers
